In [4]:
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

# --- CONFIGURATIONS ---
MAZE_ROWS = 13
MAZE_COLS = 13
STATE_SIZE = 4      # UP, DOWN, LEFT, RIGHT status
ACTION_SIZE = 4     # 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT
NUM_ENVIRONMENTS = 10  # 10 Mazes Parallel Chalenge

# =====================================================================
# 1. PARALLEL ENVIRONMENT CLASS
# =====================================================================
class ParallelMazeEnvironment:
    def __init__(self, num_envs=10):
        self.num_envs = num_envs
        self.start_pos = (0, 11)
        self.target_pos = (12, 6)

        # Har environment ke liye alag arrays
        self.mazes = np.zeros((num_envs, MAZE_ROWS, MAZE_COLS), dtype=int)
        self.agent_positions = np.zeros((num_envs, 2), dtype=int)
        self.scores = np.zeros(num_envs, dtype=int)
        self.punish_counts = np.zeros(num_envs, dtype=int)
        self.steps = np.zeros(num_envs, dtype=int)

        self.reset_all()

    def generate_single_maze(self):
        maze = np.zeros((MAZE_ROWS, MAZE_COLS), dtype=int)
        for r in range(MAZE_ROWS):
            for c in range(MAZE_COLS):
                if random.random() < 0.3:
                    maze[r, c] = 1
        maze[self.start_pos[0], self.start_pos[1]] = 0
        maze[self.target_pos[0], self.target_pos[1]] = 0
        return maze

    def reset_all(self):
        for i in range(self.num_envs):
            self.reset_single_env(i)
        return self.get_all_states()

    def reset_single_env(self, env_idx):
        self.mazes[env_idx] = self.generate_single_maze()
        self.agent_positions[env_idx] = list(self.start_pos)
        self.scores[env_idx] = 0
        self.punish_counts[env_idx] = 0
        self.steps[env_idx] = 0

    def get_all_states(self):
        states = []
        for i in range(self.num_envs):
            r, c = self.agent_positions[i]
            maze = self.mazes[i]

            up = 1 if r == 0 or maze[r-1, c] == 1 else 0
            down = 1 if r == MAZE_ROWS-1 or maze[r+1, c] == 1 else 0
            left = 1 if c == 0 or maze[r, c-1] == 1 else 0
            right = 1 if c == MAZE_COLS-1 or maze[r, c+1] == 1 else 0

            states.append([up, down, left, right])
        return np.array(states, dtype=np.float32)

    def step(self, actions):
        rewards = np.zeros(self.num_envs, dtype=np.float32)
        dones = np.zeros(self.num_envs, dtype=bool)

        for i in range(self.num_envs):
            self.steps[i] += 1
            r, c = self.agent_positions[i]
            new_r, new_c = r, c
            action = actions[i]

            if action == 0: new_r -= 1
            elif action == 1: new_r += 1
            elif action == 2: new_c -= 1
            elif action == 3: new_c += 1

            hit_wall = False
            if new_r < 0 or new_r >= MAZE_ROWS or new_c < 0 or new_c >= MAZE_COLS or self.mazes[i, new_r, new_c] == 1:
                hit_wall = True
            else:
                self.agent_positions[i] = [new_r, new_c]

            # Rules logic
            if hit_wall:
                rewards[i] = -1
                self.scores[i] -= 1
            elif tuple(self.agent_positions[i]) == self.target_pos:
                rewards[i] = 100
                self.scores[i] += 100
                dones[i] = True
            else:
                rewards[i] = 1
                self.scores[i] += 1

            if self.scores[i] >= 50 and not dones[i]:
                self.scores[i] -= 25
                self.punish_counts[i] += 1
                rewards[i] = -25

            if self.steps[i] >= 200:
                dones[i] = True

        next_states = self.get_all_states()
        return next_states, rewards, dones

# =====================================================================
# 2. DQN CLASS (Neural Network)
# =====================================================================
class DQN_Network(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN_Network, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# =====================================================================
# 3. PARALLEL AGENT CLASS
# =====================================================================
class ParallelDQNAgent:
    def __init__(self):
        self.memory = deque(maxlen=10000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.batch_size = 64

        self.model = DQN_Network(STATE_SIZE, ACTION_SIZE)
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        self.criterion = nn.MSELoss()

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, states):
        actions = []
        for i in range(NUM_ENVIRONMENTS):
            if np.random.rand() <= self.epsilon:
                actions.append(random.randrange(ACTION_SIZE))
            else:
                state_t = torch.FloatTensor(states[i]).unsqueeze(0)
                with torch.no_grad():
                    q_values = self.model(state_t)
                actions.append(torch.argmax(q_values).item())
        return np.array(actions)

    # [FIXED REPLAY FUNCTION]
    def replay(self):
        if len(self.memory) < self.batch_size:
            return

        minibatch = random.sample(self.memory, self.batch_size)

        for state, action, reward, next_state, done in minibatch:
            state_t = torch.FloatTensor(state)
            next_state_t = torch.FloatTensor(next_state)

            target = float(reward)
            if not done:
                target = float(reward + self.gamma * torch.max(self.model(next_state_t)).item())

            current_q = self.model(state_t)
            target_f = current_q.clone().detach()

            # Explicit type casting to int to fix indexing error
            target_f[int(action)] = target

            self.optimizer.zero_grad()
            loss = self.criterion(current_q, target_f)
            loss.backward()
            self.optimizer.step()

    def save_model(self, name):
        torch.save(self.model.state_dict(), name)

# =====================================================================
# 4. EXECUTION LOOP (Parallel Training)
# =====================================================================
if __name__ == "__main__":
    env = ParallelMazeEnvironment(num_envs=NUM_ENVIRONMENTS)
    agent = ParallelDQNAgent()
    total_episodes = 200

    print(f"AI Parallel Training Shuru ({NUM_ENVIRONMENTS} Mazes Aik Sath)...\n")

    states = env.reset_all()

    for episode in range(1, total_episodes + 1):
        dones = np.zeros(NUM_ENVIRONMENTS, dtype=bool)

        while not np.all(dones):
            actions = agent.act(states)
            next_states, rewards, current_dones = env.step(actions)

            for i in range(NUM_ENVIRONMENTS):
                if not dones[i]:
                    agent.remember(states[i], actions[i], rewards[i], next_states[i], current_dones[i])

            agent.replay()

            states = next_states
            dones = np.logical_or(dones, current_dones)

        if agent.epsilon > agent.epsilon_min:
            agent.epsilon *= agent.epsilon_decay

        if episode % 10 == 0:
            avg_score = np.mean(env.scores)
            total_punishes = np.sum(env.punish_counts)
            print(f"Episode Batch {episode}/{total_episodes} | Avg Score: {avg_score:.2f} | Total Batch Punishes: {total_punishes} | Epsilon: {agent.epsilon:.4f}")

        states = env.reset_all()

    agent.save_model("parallel_smart_agent.pth")
    print("\nParallel Training Khatam! Model 'parallel_smart_agent.pth' save ho gaya.")

AI Parallel Training Shuru (10 Mazes Aik Sath)...

Episode Batch 10/200 | Avg Score: 22.70 | Total Batch Punishes: 11 | Epsilon: 0.9511
Episode Batch 20/200 | Avg Score: 25.40 | Total Batch Punishes: 14 | Epsilon: 0.9046
Episode Batch 30/200 | Avg Score: 13.00 | Total Batch Punishes: 8 | Epsilon: 0.8604
Episode Batch 40/200 | Avg Score: 38.00 | Total Batch Punishes: 20 | Epsilon: 0.8183
Episode Batch 50/200 | Avg Score: 30.00 | Total Batch Punishes: 18 | Epsilon: 0.7783
Episode Batch 60/200 | Avg Score: 23.60 | Total Batch Punishes: 16 | Epsilon: 0.7403
Episode Batch 70/200 | Avg Score: 37.00 | Total Batch Punishes: 22 | Epsilon: 0.7041
Episode Batch 80/200 | Avg Score: 28.40 | Total Batch Punishes: 20 | Epsilon: 0.6696
Episode Batch 90/200 | Avg Score: 35.80 | Total Batch Punishes: 26 | Epsilon: 0.6369
Episode Batch 100/200 | Avg Score: 9.20 | Total Batch Punishes: 20 | Epsilon: 0.6058
Episode Batch 110/200 | Avg Score: 65.20 | Total Batch Punishes: 145 | Epsilon: 0.5762
Episode Batch